# LeetCode 1961 — Check If String Is a Prefix of Array bằng Python

**Link đề:** https://leetcode.com/problems/check-if-string-is-a-prefix-of-array/
**Độ khó:** Easy · **Chủ đề:** Array, Two Pointers, String

```python
class Solution:
    def isPrefixString(self, s: str, words: List[str]) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu đúng khái niệm **prefix string của mảng**: ghép **trọn vẹn** `k` từ đầu tiên (không được cắt giữa từ).
2. Biết nối chuỗi, slicing `words[:k]`, `"".join`, `str.startswith(prefix, start)`.
3. Đi từ cách "thử mọi `k`" → nối dần và **dừng sớm** → **hai con trỏ** so sánh không tạo chuỗi mới.
4. Trace từng bước, chứng minh, Time/Space; nhận ra bẫy "`s` kết thúc giữa một từ".
5. Luyện các biến thể (tìm `k`, prefix của từ, LeetCode 1662…).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho chuỗi `s` và mảng chuỗi `words`. `s` là **prefix string** của `words` nếu `s` bằng việc **nối `k` chuỗi đầu tiên** của `words` với một số nguyên dương `k ≤ len(words)`.

Trả `True` nếu `s` là prefix string của `words`, ngược lại `False`.

| `s` | `words` | Output | Giải thích |
|---|---|:---:|---|
| `"iloveleetcode"` | `["i","love","leetcode","apples"]` | `True` | `"i" + "love" + "leetcode"` (k = 3) |
| `"iloveleetcode"` | `["apples","i","love","leetcode"]` | `False` | phải bắt đầu từ `words[0]` = `"apples"` |

**Constraints:** `1 <= words.length <= 100`, `1 <= words[i].length <= 20`, `1 <= s.length <= 1000`, chỉ chữ thường.

**Bẫy quan trọng:** `s = "ilovele"`, `words = ["i","love","leetcode"]` → `False`, vì `"ilovele"` dừng **giữa** từ `"leetcode"`. "Prefix của chuỗi ghép" chưa đủ — phải khớp **ranh giới từ**.

In [1]:
words = ["i", "love", "leetcode", "apples"]
print("i" + "love" + "leetcode")
print("".join(words).startswith("ilovele"))   # True — nhưng đáp án của bài lại là False!

iloveleetcode
True


## 2. Nền tảng Python cần dùng

### 2.1. Nối chuỗi `+` và `"".join`

`a + b` tạo chuỗi mới gồm `a` rồi `b`. `"".join(list_chuoi)` nối cả list một lần, hiệu quả hơn khi có nhiều phần.

In [2]:
print("i" + "love")
print("".join(["i", "love", "leetcode"]))
print("".join([]))          # chuỗi rỗng

ilove
iloveleetcode



### 2.2. Slicing list `words[:k]`

`words[:k]` là list `k` phần tử đầu (từ chỉ số 0 tới `k-1`). Nếu `k` vượt độ dài thì chỉ lấy hết, không lỗi.

In [3]:
words = ["i", "love", "leetcode", "apples"]
for k in range(1, len(words) + 1):
    print(k, words[:k], "->", "".join(words[:k]))
print(words[:10])

1 ['i'] -> i
2 ['i', 'love'] -> ilove
3 ['i', 'love', 'leetcode'] -> iloveleetcode
4 ['i', 'love', 'leetcode', 'apples'] -> iloveleetcodeapples
['i', 'love', 'leetcode', 'apples']


### 2.3. `str.startswith(prefix, start)`

`s.startswith(p)` kiểm tra `s` có bắt đầu bằng `p`. Tham số thứ hai `start` cho phép kiểm tra **từ vị trí `start`** — tức `s[start:].startswith(p)` mà không cần tạo lát cắt.

In [4]:
s = "iloveleetcode"
print(s.startswith("ilove"))          # True
print(s.startswith("love", 1))        # True: từ vị trí 1
print(s.startswith("leetcode", 5))    # True: từ vị trí 5
print(s.startswith("leetcodex", 5))   # False: vượt quá cuối s

True
True
True
False


### 2.4. `len` và so sánh độ dài trước khi so nội dung

Hai chuỗi bằng nhau thì phải dài bằng nhau. So `len` trước là cách loại sớm rẻ tiền (`O(1)`).

In [5]:
print(len("iloveleetcode"), len("i") + len("love") + len("leetcode"))   # 13 13

13 13


## 3. Tính tay — tiền tố tích luỹ

`words = ["i","love","leetcode","apples"]`, `s = "iloveleetcode"` (dài 13):

| k | Chuỗi ghép | Độ dài | So với s |
|---:|---|---:|---|
| 1 | `i` | 1 | ngắn hơn, là tiền tố của s → tiếp |
| 2 | `ilove` | 5 | ngắn hơn, là tiền tố → tiếp |
| 3 | `iloveleetcode` | 13 | **bằng s** → `True` |

Với `s = "ilovele"` (dài 7): k = 2 → dài 5 < 7; k = 3 → dài 13 > 7 → đã **vượt** mà chưa bằng → `False`.

Nhận xét: độ dài tiền tố **tăng ngặt** theo `k` (mỗi từ dài ≥ 1), nên **nhiều nhất một** `k` có độ dài bằng `len(s)`.

## 4. Cách 1 — Baseline: thử mọi `k` (đúng theo gợi ý của đề)

Có đúng `len(words)` prefix string. Tạo từng cái và so với `s`.

In [6]:
def is_prefix_all_k(s, words):
    for k in range(1, len(words) + 1):
        if "".join(words[:k]) == s:
            return True
    return False

print(is_prefix_all_k("iloveleetcode", ["i", "love", "leetcode", "apples"]))   # True
print(is_prefix_all_k("iloveleetcode", ["apples", "i", "love", "leetcode"]))   # False
print(is_prefix_all_k("ilovele", ["i", "love", "leetcode"]))                   # False

True
False
False


### 4.1. Nút thắt của baseline

Với mỗi `k`, `"".join(words[:k])` **dựng lại từ đầu** chuỗi dài tới `20k` ký tự. Tổng chi phí ≈ `Σ 20k` = `O(m²·w)` (m = số từ, w = độ dài từ). Hai điểm lãng phí:
1. Không tận dụng chuỗi của bước trước (bước `k` chỉ khác bước `k-1` một từ).
2. Không dừng khi chuỗi ghép đã **dài hơn** `s` — từ đó trở đi không thể bằng `s` nữa.

In [7]:
# Đếm số ký tự bị dựng lại bởi baseline với 100 từ dài 20
words_big = ["a" * 20] * 100
built = sum(len("".join(words_big[:k])) for k in range(1, 101))
print("baseline dựng", built, "ký tự; nối dần chỉ cần", 20 * 100)

baseline dựng 101000 ký tự; nối dần chỉ cần 2000


## 5. Cải tiến — nối dần và dừng sớm

Giữ biến `cur` = chuỗi ghép hiện tại. Mỗi bước `cur += word`:
- nếu `cur == s` → `True`;
- nếu `len(cur) >= len(s)` (mà chưa bằng) → `False` (dừng sớm);
- hết từ mà vẫn ngắn hơn `s` → `False`.

Còn tốt hơn: **không cần tạo `cur` thật**. Chỉ cần so từng từ với **đúng đoạn tương ứng** trong `s` — đó là **hai con trỏ**: con trỏ `i` trên `s`, con trỏ duyệt `words`.

## 6. Lời giải chính — hai con trỏ, giảng từng đoạn

**Đoạn 1:** `i = 0` — vị trí trong `s` mà từ tiếp theo phải bắt đầu.

**Đoạn 2 — với mỗi `word`:** kiểm tra `s.startswith(word, i)`. Nếu không khớp (sai ký tự, **hoặc** `word` thò ra ngoài cuối `s`) → `False`. Nếu khớp → `i += len(word)`.

**Đoạn 3 — sau mỗi từ:** nếu `i == len(s)` → vừa khớp trọn `s` tại ranh giới từ → `True`.

**Đoạn 4:** hết từ mà `i < len(s)` → `False`.

In [8]:
# Đoạn 2: vì sao startswith(word, i) cũng bắt được trường hợp "thò ra ngoài"
s = "ilovele"
print(s.startswith("leetcode", 5))   # False: s[5:] = "le" ngắn hơn "leetcode"
print(s[5:], len(s[5:]))

False
le 2


In [9]:
def is_prefix_two_pointers(s, words):
    i = 0                                   # Đoạn 1
    for word in words:
        if not s.startswith(word, i):       # Đoạn 2
            return False
        i += len(word)
        if i == len(s):                     # Đoạn 3
            return True
    return False                            # Đoạn 4

print(is_prefix_two_pointers("iloveleetcode", ["i", "love", "leetcode", "apples"]))   # True
print(is_prefix_two_pointers("iloveleetcode", ["apples", "i", "love", "leetcode"]))   # False
print(is_prefix_two_pointers("ilovele", ["i", "love", "leetcode"]))                   # False
print(is_prefix_two_pointers("ilove", ["i"]))                                         # False: hết từ

True
False
False
False


## 7. TRACE từng bước

Ví dụ 1: `s = "iloveleetcode"` (len 13), `words = ["i","love","leetcode","apples"]`.

| bước | word | i trước | `s.startswith(word, i)` | i sau | i == 13? | Hành động |
|---:|---|---:|:---:|---:|:---:|---|
| 1 | i | 0 | True | 1 | không | tiếp |
| 2 | love | 1 | True | 5 | không | tiếp |
| 3 | leetcode | 5 | True | 13 | **có** | trả `True` (không cần xem `"apples"`) |

Ví dụ bẫy: `s = "ilovele"` (len 7): bước 3 `startswith("leetcode", 5)` là `False` → trả `False`.

In [10]:
def trace_is_prefix(s, words):
    print(f"s = {s!r} (len {len(s)}), words = {words}")
    i = 0
    for step, word in enumerate(words, start=1):
        ok = s.startswith(word, i)
        print(f"  bước {step}: word={word!r:<11} i={i:>2} đoạn s tương ứng={s[i:i + len(word)]!r:<11} khớp? {ok}")
        if not ok:
            print("  -> không khớp -> False")
            return False
        i += len(word)
        if i == len(s):
            print(f"  -> i = {i} = len(s): khớp trọn tại ranh giới từ -> True")
            return True
    print(f"  -> hết từ nhưng i = {i} < len(s) = {len(s)} -> False")
    return False

trace_is_prefix("iloveleetcode", ["i", "love", "leetcode", "apples"])
print()
trace_is_prefix("ilovele", ["i", "love", "leetcode"])
print()
trace_is_prefix("ilovex", ["i", "love"])

s = 'iloveleetcode' (len 13), words = ['i', 'love', 'leetcode', 'apples']
  bước 1: word='i'         i= 0 đoạn s tương ứng='i'         khớp? True
  bước 2: word='love'      i= 1 đoạn s tương ứng='love'      khớp? True
  bước 3: word='leetcode'  i= 5 đoạn s tương ứng='leetcode'  khớp? True
  -> i = 13 = len(s): khớp trọn tại ranh giới từ -> True

s = 'ilovele' (len 7), words = ['i', 'love', 'leetcode']
  bước 1: word='i'         i= 0 đoạn s tương ứng='i'         khớp? True
  bước 2: word='love'      i= 1 đoạn s tương ứng='love'      khớp? True
  bước 3: word='leetcode'  i= 5 đoạn s tương ứng='le'        khớp? False
  -> không khớp -> False

s = 'ilovex' (len 6), words = ['i', 'love']
  bước 1: word='i'         i= 0 đoạn s tương ứng='i'         khớp? True
  bước 2: word='love'      i= 1 đoạn s tương ứng='love'      khớp? True
  -> hết từ nhưng i = 5 < len(s) = 6 -> False


False

`enumerate(words, start=1)` đánh số bắt đầu từ 1 thay vì 0 — tiện cho in "bước 1, 2, 3".

## 8. Các cách giải khác

### 8.1. Cách 2 — nối dần với `cur`, dừng sớm theo độ dài

In [11]:
def is_prefix_concat(s, words):
    cur = ""
    for word in words:
        cur += word
        if len(cur) >= len(s):          # đã đủ dài: hoặc bằng s, hoặc hỏng
            return cur == s
    return False

print(is_prefix_concat("iloveleetcode", ["i", "love", "leetcode", "apples"]),
      is_prefix_concat("ilovele", ["i", "love", "leetcode"]))

True False


### 8.2. Cách 3 — tìm `k` bằng độ dài tích luỹ, rồi so một lần

Vì chỉ **tối đa một** `k` có tổng độ dài bằng `len(s)`, ta cộng dồn độ dài tới khi `total >= len(s)`. Nếu `total == len(s)` thì so `"".join(words[:k]) == s` đúng **một lần**.

In [12]:
def is_prefix_by_length(s, words):
    total = 0
    for k, word in enumerate(words, start=1):
        total += len(word)
        if total == len(s):
            return "".join(words[:k]) == s
        if total > len(s):
            return False
    return False

print(is_prefix_by_length("iloveleetcode", ["i", "love", "leetcode", "apples"]),
      is_prefix_by_length("iloveleetcode", ["apples", "i", "love", "leetcode"]))

True False


### 8.3. Cách 4 — con trỏ ký tự-từng-ký tự (không dùng `startswith`)

Phiên bản "thuần" hai con trỏ, hữu ích khi phỏng vấn yêu cầu không dùng hàm có sẵn.

In [13]:
def is_prefix_char_pointers(s, words):
    i = 0
    for word in words:
        for ch in word:
            if i == len(s) or s[i] != ch:   # s hết giữa chừng từ, hoặc ký tự khác
                return False
            i += 1
        if i == len(s):
            return True
    return False

print(is_prefix_char_pointers("iloveleetcode", ["i", "love", "leetcode", "apples"]),
      is_prefix_char_pointers("ilovele", ["i", "love", "leetcode"]))

True False


## 9. Code nộp LeetCode (Python 3)

Trên LeetCode, `List` đã được import sẵn; khi chạy cục bộ ta tự `from typing import List`.

In [14]:
from typing import List

class Solution:
    def isPrefixString(self, s: str, words: List[str]) -> bool:
        i = 0
        for word in words:
            if not s.startswith(word, i):
                return False
            i += len(word)
            if i == len(s):
                return True
        return False


sol = Solution()
assert sol.isPrefixString("iloveleetcode", ["i", "love", "leetcode", "apples"]) is True
assert sol.isPrefixString("iloveleetcode", ["apples", "i", "love", "leetcode"]) is False
print("2 ví dụ của đề PASS")

2 ví dụ của đề PASS


## 10. Vì sao thuật toán đúng?

**Invariant:** trước khi xét từ thứ `j` (0-based), ta có `s[:i] == words[0] + … + words[j-1]` và `i < len(s)`.

- Ban đầu `j = 0`, `i = 0`: cả hai vế rỗng — đúng.
- Xét `words[j]`: `s.startswith(words[j], i)` đúng ⇔ `s[i : i+len(words[j])] == words[j]` **và** đoạn đó nằm trọn trong `s`. Nếu đúng, sau `i += len(words[j])` ta có `s[:i] == words[0] + … + words[j]` — invariant giữ.
- Nếu sai: tiền tố `k = j+1` (và mọi `k` lớn hơn, vì chúng chứa tiền tố này) không thể bằng `s` hoặc là tiền tố của `s`; còn các `k ≤ j` đều cho chuỗi **ngắn hơn** `s` (vì `i < len(s)`). Vậy không `k` nào thoả → `False` đúng.
- Nếu `i == len(s)`: `s == words[0] + … + words[j]` với `k = j+1 ≥ 1` → `True` đúng.
- Hết từ mà `i < len(s)`: mọi `k` đều cho chuỗi ngắn hơn `s` → `False` đúng.

Điểm mấu chốt: `True` chỉ được trả **ngay sau khi trọn một từ** — đó là cách thuật toán tôn trọng ranh giới từ.

## 11. Time / Space Complexity

Gọi `n = len(s)`, `m = len(words)`, `w` = độ dài từ lớn nhất (≤ 20).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Thử mọi `k`, dựng lại | `O(m² · w)` | `O(m · w)` | đúng gợi ý của đề, lãng phí |
| 2. Nối dần + dừng sớm | `O(n)` ký tự nối* | `O(n)` | đơn giản |
| 3. Độ dài tích luỹ + so một lần | `O(m + n)` | `O(n)` | so nội dung đúng một lần |
| 4. Con trỏ ký tự | `O(min(n, tổng độ dài words))` | `O(1)` | không tạo chuỗi |
| 5. `startswith` (chính) | `O(min(n, tổng độ dài words))` | `O(1)` | nên nộp |

\*Cách 2 dừng khi `len(cur) ≥ n`, nên `cur` không dài quá `n + w`; nhưng `cur += word` có thể sao chép chuỗi mỗi lần — tệ nhất `O(n · m)` nếu trình thông dịch không tối ưu.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.isPrefixString("iloveleetcode", ["i", "love", "leetcode", "apples"]), True)
check("đề 2", sol.isPrefixString("iloveleetcode", ["apples", "i", "love", "leetcode"]), False)
check("dừng giữa từ", sol.isPrefixString("ilovele", ["i", "love", "leetcode"]), False)
check("s dài hơn tổng words", sol.isPrefixString("ilovex", ["i", "love"]), False)
check("k = 1", sol.isPrefixString("abc", ["abc", "d"]), True)
check("k = len(words)", sol.isPrefixString("abcd", ["ab", "cd"]), True)
check("s ngắn hơn từ đầu", sol.isPrefixString("a", ["aa", "a"]), False)
check("sai ở ký tự cuối", sol.isPrefixString("abd", ["ab", "c"]), False)
check("từ lặp", sol.isPrefixString("aaa", ["a", "a", "a", "a"]), True)

PASS đề 1: True
PASS đề 2: False
PASS dừng giữa từ: False
PASS s dài hơn tổng words: False
PASS k = 1: True
PASS k = len(words): True
PASS s ngắn hơn từ đầu: False
PASS sai ở ký tự cuối: False
PASS từ lặp: True


In [16]:
# Đối chiếu các cách trên dữ liệu ngẫu nhiên có seed
import random
random.seed(1961)
funcs = [is_prefix_all_k, is_prefix_two_pointers, is_prefix_concat, is_prefix_by_length,
         is_prefix_char_pointers, Solution().isPrefixString]
trues = 0
for _ in range(2000):
    words = ["".join(random.choice("ab") for _ in range(random.randint(1, 3))) for _ in range(random.randint(1, 5))]
    if random.random() < 0.5:
        s = "".join(words[:random.randint(1, len(words))])          # tạo ca True
    else:
        s = "".join(random.choice("ab") for _ in range(random.randint(1, 8)))
    results = {f(s, words) for f in funcs}
    assert len(results) == 1, (s, words, results)
    trues += results.pop()
print("2000 bộ ngẫu nhiên: mọi cách khớp nhau; số ca True =", trues)

2000 bộ ngẫu nhiên: mọi cách khớp nhau; số ca True = 1061


## 13. Sai lầm người mới hay mắc

### Lỗi 1 — Chỉ kiểm tra `"".join(words).startswith(s)`

In [17]:
def buggy_startswith_join(s, words):
    return "".join(words).startswith(s)

print("buggy:", buggy_startswith_join("ilovele", ["i", "love", "leetcode"]),
      "| đúng:", is_prefix_two_pointers("ilovele", ["i", "love", "leetcode"]))

buggy: True | đúng: False


Bỏ qua yêu cầu ghép **trọn từ**. Sửa: phải kiểm tra `s` kết thúc đúng tại ranh giới một từ.

### Lỗi 2 — Kiểm tra `s` có chứa các từ (không theo thứ tự / không từ đầu)

```python
return all(w in s for w in words[:3])   # SAI: không đảm bảo bắt đầu từ words[0], đúng thứ tự, liền nhau
```
Ví dụ 2 của đề (`"apples"` đứng đầu) cho thấy thứ tự và điểm bắt đầu là bắt buộc.

### Lỗi 3 — Trả `True` khi `s` là **tiền tố của từ đầu tiên**

In [18]:
def buggy_prefix_of_word(s, words):
    i = 0
    for word in words:
        j = 0
        while j < len(word) and i < len(s) and s[i] == word[j]:
            i += 1
            j += 1
        if i == len(s):
            return True            # SAI: có thể đang dừng GIỮA word (j < len(word))
        if j < len(word):
            return False
    return False

print("buggy:", buggy_prefix_of_word("a", ["aa", "a"]), "| đúng:", is_prefix_two_pointers("a", ["aa", "a"]))

buggy: True | đúng: False


Sửa: chỉ trả `True` khi `i == len(s)` **và** `j == len(word)` (đã hết trọn từ).

### Lỗi 4 — Quên trường hợp hết `words` mà `s` vẫn còn

Nếu vòng lặp kết thúc mà không `return`, hàm Python trả `None` — LeetCode coi là sai kiểu/sai đáp án. Luôn có `return False` cuối hàm.

In [19]:
def buggy_no_final_return(s, words):
    i = 0
    for word in words:
        if not s.startswith(word, i):
            return False
        i += len(word)
        if i == len(s):
            return True
    # quên return False

print(repr(buggy_no_final_return("ilovex", ["i", "love"])))   # None

None


### Lỗi 5 — Dùng `s[i:].startswith(word)` trong vòng lặp

Đúng, nhưng mỗi lần `s[i:]` **tạo lát cắt mới** tốn `O(n)` → tổng `O(n·m)`. `s.startswith(word, i)` không tạo bản sao.

## 14. Mẹo nhận dạng pattern

- "Ghép các phần tử **đầu tiên** của mảng" → tiền tố tích luỹ; độ dài tăng ngặt → **dừng sớm** khi vượt.
- "So một chuỗi với dãy mảnh" → **hai con trỏ**: một trên chuỗi lớn, một trên danh sách mảnh; tránh dựng chuỗi trung gian.
- Có điều kiện "trọn vẹn"/"ranh giới" → chỉ được kết luận `True` tại thời điểm **vừa xong một mảnh**.
- Cùng họ: 1662 (Check If Two String Arrays are Equivalent), 14 (Longest Common Prefix), 2255 (Count Prefixes of a Given String), 1455 (Check If a Word Occurs As a Prefix of Any Word in a Sentence).

## 15. Biến thể

### 15.1. Trả về `k` (hoặc `-1` nếu không có)

In [20]:
def prefix_k(s, words):
    i = 0
    for k, word in enumerate(words, start=1):
        if not s.startswith(word, i):
            return -1
        i += len(word)
        if i == len(s):
            return k
    return -1

print(prefix_k("iloveleetcode", ["i", "love", "leetcode", "apples"]), prefix_k("ilovele", ["i", "love", "leetcode"]))

3 -1


### 15.2. LeetCode 1662 — hai mảng chuỗi có ghép ra cùng một chuỗi không?

Có thể so `"".join(a) == "".join(b)`; cách tiết kiệm bộ nhớ là dùng **generator** sinh từng ký tự rồi so bằng `zip` (kèm kiểm tra tổng độ dài).

In [21]:
def array_strings_are_equal(word1, word2):
    if sum(map(len, word1)) != sum(map(len, word2)):
        return False
    chars1 = (ch for w in word1 for ch in w)   # generator: sinh dần, không tạo chuỗi lớn
    chars2 = (ch for w in word2 for ch in w)
    return all(a == b for a, b in zip(chars1, chars2))

print(array_strings_are_equal(["ab", "c"], ["a", "bc"]), array_strings_are_equal(["a", "cb"], ["ab", "c"]))

True False


`map(len, word1)` áp `len` cho từng phần tử; `zip(x, y)` ghép đôi từng phần tử của hai dãy, dừng khi dãy ngắn hơn hết (vì vậy cần so độ dài trước).

### 15.3. Nới lỏng: được phép dừng giữa từ

Khi đó bài trở thành "`s` có phải tiền tố của chuỗi ghép không" — chính là "lỗi 1" ở trên, nhưng là **đáp án đúng** của biến thể này. Đọc kỹ đề là kỹ năng số một!

In [22]:
def is_prefix_of_concatenation(s, words):
    i = 0
    for word in words:
        take = min(len(word), len(s) - i)
        if s[i:i + take] != word[:take]:
            return False
        i += take
        if i == len(s):
            return True
    return False

print(is_prefix_of_concatenation("ilovele", ["i", "love", "leetcode"]))   # True ở biến thể này

True


## 16. Bài tập (TODO — tự làm, không chạy tự động)

**Bài 1 (dự đoán output):** `"abc".startswith("bc", 1)`, `"abc".startswith("bcd", 1)`, `["x","y","z"][:2]`, `"".join([])`.

**Bài 2 (tính tay):** `s = "abcde"`, `words = ["ab","cd","e","f"]` và `words = ["ab","cde"]` và `words = ["abc","d"]`. Kết quả mỗi ca? Ca nào trả lời ở bước mấy?

**Bài 3 (điền chỗ trống):** hoàn thiện lời giải hai con trỏ.

In [23]:
def exercise_is_prefix(s, words):
    i = 0
    for word in words:
        # TODO 1: nếu word không khớp s từ vị trí i -> False
        # TODO 2: tăng i
        # TODO 3: nếu i == len(s) -> True
        pass
    return False

# assert exercise_is_prefix("iloveleetcode", ["i", "love", "leetcode", "apples"]) is True
# assert exercise_is_prefix("ilovele", ["i", "love", "leetcode"]) is False

**Bài 4 (tự viết lại):** viết `exercise_is_prefix_no_builtin(s, words)` so **từng ký tự**, không dùng `startswith`, không slicing.

In [24]:
def exercise_is_prefix_no_builtin(s, words):
    # TODO
    pass

# assert exercise_is_prefix_no_builtin("abcd", ["ab", "cd"]) is True
# assert exercise_is_prefix_no_builtin("a", ["aa"]) is False

**Bài 5 (biến thể):** viết `exercise_count_prefix_words(words, s)` (LeetCode 2255): đếm số chuỗi trong `words` là **tiền tố** của `s`.

**Bài 6 (mini challenge):** viết `exercise_longest_prefix_k(s, words)` trả `k` **lớn nhất** sao cho ghép `k` từ đầu là **tiền tố** (không cần bằng) của `s`; trả `0` nếu từ đầu tiên đã không khớp.

In [25]:
def exercise_count_prefix_words(words, s):
    # TODO
    pass

def exercise_longest_prefix_k(s, words):
    # TODO
    pass

# assert exercise_count_prefix_words(["a", "b", "c", "ab", "bc", "abc"], "abc") == 3
# assert exercise_longest_prefix_k("iloveleetcode", ["i", "love", "leet", "x"]) == 3
# assert exercise_longest_prefix_k("abc", ["b"]) == 0

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | `start` dịch điểm bắt đầu | `"abc"[1:]` là `"bc"` | `"bcd"` dài hơn phần còn lại |
| 2 | theo dõi `i` | ca 1: i = 2, 4, 5 | ca 3: `"abc"` rồi `"d"` → i = 4 < 5, hết từ |
| 3 | 3 bước mỗi từ | `s.startswith(word, i)` | `i += len(word)`; `if i == len(s): return True` |
| 4 | con trỏ `i` trên `s` | vòng trong qua `ch in word` | `if i == len(s) or s[i] != ch: return False` |
| 5 | mỗi từ độc lập | `s.startswith(w)` | `sum(s.startswith(w) for w in words)` |
| 6 | giống bài 3 nhưng đếm | không yêu cầu `i == len(s)` | dừng khi không khớp; trả số từ đã khớp |

## 18. Đáp án tham khảo — chỉ xem sau khi tự làm

**Bài 1:** `True`, `False`, `['x', 'y']`, `''`.

**Bài 2:** `["ab","cd","e","f"]` → `True` ở bước 3; `["ab","cde"]` → `True` ở bước 2; `["abc","d"]` → `False` (hết từ khi `i = 4`).

In [26]:
print("abc".startswith("bc", 1), "abc".startswith("bcd", 1), ["x", "y", "z"][:2], repr("".join([])))
for ws in [["ab", "cd", "e", "f"], ["ab", "cde"], ["abc", "d"]]:
    print(ws, is_prefix_two_pointers("abcde", ws), "k =", prefix_k("abcde", ws))


def answer_is_prefix(s, words):
    i = 0
    for word in words:
        if not s.startswith(word, i):
            return False
        i += len(word)
        if i == len(s):
            return True
    return False


def answer_is_prefix_no_builtin(s, words):
    i = 0
    for word in words:
        for ch in word:
            if i == len(s) or s[i] != ch:
                return False
            i += 1
        if i == len(s):
            return True
    return False


def answer_count_prefix_words(words, s):
    return sum(s.startswith(w) for w in words)


def answer_longest_prefix_k(s, words):
    i, k = 0, 0
    for word in words:
        if not s.startswith(word, i):
            break
        i += len(word)
        k += 1
    return k


assert answer_is_prefix("iloveleetcode", ["i", "love", "leetcode", "apples"]) is True
assert answer_is_prefix("ilovele", ["i", "love", "leetcode"]) is False
assert answer_is_prefix_no_builtin("abcd", ["ab", "cd"]) is True
assert answer_is_prefix_no_builtin("a", ["aa"]) is False
assert answer_count_prefix_words(["a", "b", "c", "ab", "bc", "abc"], "abc") == 3
assert answer_longest_prefix_k("iloveleetcode", ["i", "love", "leet", "x"]) == 3
assert answer_longest_prefix_k("abc", ["b"]) == 0
print("Đáp án tham khảo PASS toàn bộ test!")

True False ['x', 'y'] ''
['ab', 'cd', 'e', 'f'] True k = 3
['ab', 'cde'] True k = 2
['abc', 'd'] False k = -1
Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def isPrefixString(self, s: str, words: List[str]) -> bool:
        i = 0
        for w in words:
            if not s.startswith(w, i):   # sai ký tự hoặc thò ra ngoài s
                return False
            i += len(w)
            if i == len(s):              # chỉ kết luận True ở RANH GIỚI từ
                return True
        return False
```

**Câu thần chú:** *con trỏ trên `s`, khớp trọn từng từ, bằng độ dài đúng lúc hết một từ mới là True.*

**Checklist trước khi submit:**
- [ ] `"ilovele"` với `["i","love","leetcode"]` trả `False` (dừng giữa từ)?
- [ ] `s` dài hơn tổng các từ trả `False`, và có `return False` cuối hàm (không trả `None`)?
- [ ] Bắt đầu so từ `words[0]`, đúng thứ tự?
- [ ] Không tạo lát cắt `s[i:]` lặp lại trong vòng (dùng `startswith(w, i)`)?